---
title: Relatório de mineração de dados
author: Erick Frederich Gomes Lima
format:
  pdf:
    colorlinks: true
    linkcolor: black
    toccolor: black
    urlcolor: blue
lang: pt
toc: true
toc-title: Sumário
toc-depth: 3
execute:
  echo: false
editor: visual
csl: normas_abnt.csl
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.5
  kernelspec:
    display_name: Python 3 (ipykernel)
    language: python
    name: python3
---

## 

In [ ]:
#| include: false
%matplotlib inline

In [ ]:
#| include: false
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\Erick\Desktop\Estudo\UFS\MINERAÇÃO DE DADOS\Registros.csv",sep=';')

In [ ]:
#| include: false
sns.set_style()
sns.set_theme(style="darkgrid")  

# Introdução

A base de dados a ser analisada é de solicitações de serviçoes municipais de Vancouver no Canadá. A base contém 1287722 registros detalhados das demandas solicitadas pela população das mais diversas categorias, quais os departamentos responsáveis, o status do caso, entre outras variáveis.

Cada registro na base representa uma solicitação individual, com data e hora de abertura e fechamento do caso, os motivos de fechamento, geolocalização, entre outras informações úteis para a análise.

Na primeira parte do relatório fiz uma análise dos departamentos - divisões com maior número de solicitações, analisando o tempo médio ate encerramento dos casos dentre outras coisas.

Para a segunda part do relatório foi escolhido somente o departamento "ENG - Streets Operations" para ser analisado, um departamento responsável pela manutenção das vias urbanas. A amostra selecionada compreende todos os chamados voltados para a operação, conservação e reparo de vias públicas da cidade. Através desse recorte, investigamos por exemplo a demanda de solicitações direcionadas ao departamento, quais os tipos de solicitação mais frequentes, proporção de casos encerrados e abertos, quais os principais motivos de encerramento dos casos, qual o tempo médio do encerramento dos casos, qual a média em dias de fechamento de caso para cada categoria, entre outras análises

# Exploração inicial dos dados

Abaixo podemos ver todas as variáveis disponiveis na base de dados

In [ ]:
df.info()

## Departamentos e Divisões

In [ ]:
df['Department'].apply(lambda x:x.split(' - ')[0]).unique()

Podemos ver que existem 16 departamentos, sendo algumas de suas divisões as seguintes:

In [ ]:
tbl_dept = (df['Department'].value_counts()
    .reset_index()
    .set_axis(['Departamento - Divisão','Número de solicitações'],axis=1).head(10))

(tbl_dept.style
   .set_caption("Departamentos da base de dados")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

## Dados Missing

Antes de qualquer análise é necessário ver quantos dados estão faltando na nossa base de dados, como pode ser visto na tabela abaixo:

In [ ]:
df_missing = df.isnull().sum().reset_index().set_axis(['Variável','Missing'],axis=1)
(df_missing.style
   .set_caption("Dados missing")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

A maior concentração de dados missing está na parte de geolocalização, o que não atrapalhará nossa análise.

# Análise geral dos departamentos

Vamos comparar os top 10 departamentos com mais números de serviços solicitados.

In [ ]:
df_dept_top_3 = df[df['Department'].isin(['ENG - Sanitation Services','DBL - Services Centre',
                                        'ENG - Parking Enforcement and Operations','ENG - Streets Operations',
                                         '311 Contact Centre','DBL - Licence Office','PR - Urban Forestry',
                                         'DBL - Property Use Inspections','CMO - Business and Election Services',
                                         'DBL - Animal Services'])]
tbl_top10 = (df['Department'].value_counts().reset_index()
    .set_axis(['Departamento - Divisão','Número de solicitações'],axis=1).head(10))


(tbl_top10.style
   .set_caption("Top 10 departamentos com maior número de solicitações")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

## Casos fechados x abertos

Vamos comparar qual a proporção de casos abertos e fechados dos departamentos

In [ ]:
plt.figure(figsize=(10,7))
plt.title('Proporção de casos fechados e abertos do top 10',fontsize=16,fontweight='bold')    
ax12 = sns.countplot(y='Department',data=df_dept_top_3,hue='Status')
plt.xlabel('Quantidade de casos')
plt.tight_layout()
for container in ax12.containers:
    ax12.bar_label(container,fontweight='bold')

A análise visual demonstra que todos os 10 principais departamentos operam com uma taxa de resolução de chamados extremamente alta, evidenciando que os departamentos conseguem dar conta da quantidade de solicitações recebidas

## Tempo médio em dias de encerramento dos casos em cada um dos departamentos

In [ ]:
df_dept_top_3['Service request open timestamp'] = df_dept_top_3['Service request open timestamp'].apply(lambda x:x.split('T')[0])
#transofrmando em data
df_dept_top_3['Service request open timestamp'] = pd.to_datetime(df_dept_top_3['Service request open timestamp'])
df_dept_top_3['Service request close date'] = pd.to_datetime(df_dept_top_3['Service request close date'])

In [ ]:
df_dept_top_3['Dias ate encerramento'] = (
    df_dept_top_3['Service request close date'] - df_dept_top_3['Service request open timestamp'])
df_dept_top_3[['Service request open timestamp','Service request close date','Dias ate encerramento']]

df_dept_top_3['Dias ate encerramento'] = df_dept_top_3['Dias ate encerramento'].dt.days 

In [ ]:
tbl_dept_media = (df_dept_top_3.groupby('Department')['Dias ate encerramento'].mean().sort_values(ascending=False)
    .reset_index()
    .set_axis(['Departamento - Divisão','Dias até encerramento'],axis=1))

(tbl_dept_media.style
   .set_caption("Departamentos do top 10 com maiores médias em dias para encerramento")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

A análise do tempo médio (em dias) para o encerramento dos casos para cada departamento revela uma grande discrepância entre o primeiro lugar e os outros departamentos, o que pode ser explicado pelas particularidades da divisão PR - Urban Forestry, responsável por serviços que exigem planejamento logístico e maquinário pesado, como o manejo de árvores. A maioria dos departamentos operacionais e de atendimento, como Licence Office, Streets Operations e Sanitation Services, mantém um tempo de resposta intermediário, resolvendo os casos em uma janela de 7 a 13 dias. Em contrapartida, divisões reponsáveis por fiscalizações e ermergências demonstram alta agilidade, concluindo seus chamados em menos de 7 dias, evidenciando fluxos de trabalho projetados para resoluções rápidas e imediatas no espaço urbano.

A divisão de Animal Services é a que possui a menor média de tempo em dias para o encerramento de seus casos, o que faz sentido devido a natureza das solicitações recebidas pela divisão, que geralmente exigem ação imediata de profissionais.

## Departamentos com maior número de casos encerrados por serviço provido

In [ ]:
tbl_pro_serv = (df_dept_top_3[df_dept_top_3['Closure reason'].isin(['Service provided'])]
    .groupby('Department')['Closure reason']
    .value_counts()
    .reset_index()
    .set_axis(['Departamento - Divisão','Motivo do encerramento','Quantidade'],axis=1)
    .assign(Porcentagem=lambda x: ((x['Quantidade'] / x['Departamento - Divisão']
        .map(df_dept_top_3['Department']
        .value_counts())) * 100)
        .round(2))
    .sort_values(by='Porcentagem',ascending=False))

(tbl_pro_serv.style
   .set_caption("Departamentos com maior número de casos listados como 'Service provided'")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

A tabela evidencia que a divisão DBL - Services Centre, divisão responsável pela regularização imobiliária e de negócios, lidera isoladamente na efetividade de resolução, com 87,26% das solicitações encerradas com o status de serviço efetivamente prestado ('Service provided').

## Canais de atendimento

Principais canais de contato entre a população e os departamentos.

In [ ]:
plt.figure(figsize=(11,7))
ax10 = sns.countplot(x='Channel',data=df,hue='Channel')

for container in ax10.containers:
    ax10.bar_label(container,fontweight='bold',
                   labels=[f'{int(v.get_height()):,}\n({v.get_height()/len(df):.1%})' for v in container])

plt.title('Canais de atendimento mais utilizados',fontsize=16,fontweight='bold')    

Ao analisar os canais de atendimento, fica evidente uma forte preferência dos cidadãos por plataformas de autoatendimento digital, com o canal 'WEB' a liderar de forma destacada, registando mais de 578 mil solicitações. O contato telefônico ('Phone') ocupa a segunda posição com mais de 460 mil registos, indicando que o atendimento por voz continua a ser um meio essencial e muito procurado pela população de Vancouver, possivelmente para casos de mais urgência.

## Canais de comunicação mais usados ao longo dos anos

In [ ]:
#col ano
df['Ano'] = df['Service request open timestamp'].apply(lambda x:str(x).split("-")[0])

In [ ]:
df_canal_ano = (df.groupby(['Ano','Channel'])['Channel'].value_counts()
    .reset_index()
    .set_axis(['Ano','Canal','Quantidade'],axis=1))

In [ ]:
plt.figure(figsize=(10,6))
plt.tight_layout()
sns.lineplot(y='Quantidade',x='Ano',hue='Canal',data=df_canal_ano)
plt.legend(bbox_to_anchor=(1, 1),loc='upper left')
plt.title('Canais de comunicação ao longo dos anos',fontsize=18,fontweight='bold')

A análise temporal dos canais de comunicação demonstra que o padrão de comportamento da população se mantém estável ao longo dos anos. A plataforma 'WEB' e o telefone ('Phone') lideram consistentemente o volume de solicitações em todo o período analisado, de 2022 a 2026. A via 'WEB' permanece como o principal meio de contato em todos os anos registrados, atingindo números superiores a 120 mil solicitações anuais na maior parte do período (2022, 2023 e 2025). O canal telefônico manteve-se firme na segunda posição, oscilando em torno da marca de 100 mil chamados anuais entre 2022 e 2025. O uso do aplicativo móvel ('Mobile App') também apresenta uma base consolidada, mantendo-se invariavelmente como a terceira opção mais utilizada pelos cidadãos de Vancouver. O restante dos canais de comunicação tem baixísimo usa pelos cidadâos.

## Top 5 departamentos com mais casos ao longo dos anos

In [ ]:
df_dept_top_3['Ano'] = df_dept_top_3['Service request open timestamp'].apply(lambda x:str(x).split("-")[0])


df_dept_top_5 = df[df['Department'].isin(['ENG - Sanitation Services','DBL - Services Centre',
                                        'ENG - Parking Enforcement and Operations','ENG - Streets Operations',
                                         '311 Contact Centre'])]

df_casos_ano = (df_dept_top_5.groupby(['Ano','Department'])['Department'].value_counts()
    .reset_index()
    .set_axis(['Ano','Departamento - Divisão','Quantidade'],axis=1)
    .sort_values(by='Ano'))

In [ ]:
plt.figure(figsize=(10,6))
plt.tight_layout()
sns.lineplot(y='Quantidade',x='Ano',hue='Departamento - Divisão',data=df_casos_ano)
plt.legend(bbox_to_anchor=(1, 1),loc='upper left')
plt.title('Quantidade de solicitações por departamento',fontsize=18,fontweight='bold')

Fica evidente que a divisão 'ENG - Sanitation Services' mantém a liderança isolada de demandas em todo o período analisado, registrando um volume amplamente superior aos demais (acima de 70.000 casos anuais entre 2022 e 2024). No entanto, o gráfico demonstra uma tendência de queda acentuada nas solicitações para este departamento a partir de 2024, caindo para menos de 50.000 chamados registrados em 2026, o que é explicado pelo fato de que a base de dados não abrange o ano de 2026 na sua integridade. Em contraste, os outros quatro departamentos ('DBL - Services Centre', 'ENG - Parking Enforcement and Operations', 'ENG - Streets Operations' e '311 Contact Centre') apresentam volumes mais baixos, oscilando predominantemente na faixa de 10.000 a 30.000 solicitações anuais ao longo de todo o período.

# Departamento com maior tempo médio de encerramento dos casos (PR - Urban Forestry)

O departamento que tem a maior média de dias até encerramento dos casos é o de de parques e recreação, especificamente na divisão "Urban Forestry", que é responsável pela Silvicultura Urbana. Entre os serviços solicitados ao departamento estão:

In [ ]:
df_urban_forestry = df_dept_top_3[df_dept_top_3['Department'].isin(['PR - Urban Forestry'])]

plt.figure(figsize=(10,7))
ax7 = sns.countplot(y='Service request type',data=df_urban_forestry,hue='Service request type',
                    order=df_urban_forestry['Service request type'].value_counts().index)
plt.xlabel('Quantidade')
for container in ax7.containers:
    ax7.bar_label(container,label_type='edge',color='black',fontweight='bold',
                 labels=[f'{int(v.get_width()):,} ({v.get_width()/len(df_urban_forestry):.1%})' for v in container])

plt.title('Ranking de solicitações mais frequentes (PR - Urban Forestry)',fontsize=16,fontweight='bold')  
plt.tight_layout()

O gráfico demonstra que a demanda do departamento de Urban Forestry é massivamente concentrada em serviços de arborização urbana. A categoria 'City and Park Trees Maintenance Case' lidera isoladamente com 52.784 solicitações, representando a grande maioria de todas as demandas recebidas pelo setor. As demais categorias, referentes ao controle de roedores ('Rats and Rodents', com 1.575 casos), controle de pragas ('Pest Management Inquiry', com 542) e ninhos de gansos ('Canada Geese Nests', com 372), possuem volumes expressivamente menores. Essa disparidade evidencia que o foco operacional do departamento está voltado quase em sua totalidade para o manejo, poda e manutenção de árvores em vias e parques públicos

## Proporção de casos abertos x fechados (PR - Urban Forestry)

In [ ]:
plt.figure(figsize=(10,7))
ax8 = sns.countplot(y='Service request type',data=df_urban_forestry,hue='Status',
                    order=df_urban_forestry['Service request type'].value_counts().index)
plt.xlabel('Quantidade')
for container in ax8.containers:
    ax8.bar_label(container,label_type='edge',color='black',fontweight='bold')

plt.title('Proporção de casos abertos e fechados por tipo de serviço (PR - Urban Forestry)',fontsize=16,fontweight='bold')    
plt.tight_layout()

Apesar do elevado volume de chamados de manutenção de árvores, o departamento apresenta um índice de resolução extremamente alto. Dos 52.784 casos abertos para manutenção de árvores, 51.760 foram devidamente encerrados, permanecendo apenas 1.024 em aberto. Nas demais categorias, a taxa de pendência é praticamente nula: os casos de gansos foram 100% concluídos (372 casos), enquanto Rats and Rodents e Pest Management Inquiry mantêm apenas 4 e 16 casos abertos, respectivamente. Isso mostra que, embora o setor demande mais tempo para concluir os serviços, a resolução total das solicitações é altamente eficaz.

## Tempo médio em dias para o encerramento dos casos (PR - Urban Forestry)

Na tabela abaixo podemos ver o tempo médio em dias para o encerramento de cada tipo de serviço solicitado.

In [ ]:
tbl_media_pr = (df_urban_forestry.groupby('Service request type')['Dias ate encerramento'].mean()
    .reset_index().
    set_axis(['Tipo de serviço solicitado','Dias até encerramento'],axis=1)
    .sort_values(by='Dias até encerramento',ascending=False))

(tbl_media_pr.style
   .set_caption("Tempo médio de encerramento dos casos do departamento PR - Urban Forestry")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

A análise do tempo media de encerramento dos casos dentro da divisão Urban Forestry revela o que a categoriza como o departamento com o maior tempo médio de encerramento dentre todos. Solicitações referentes a investigação e controle de pestes (Pest Management Inquiry Case) tem média de 73 dias até encerramento dos casos, explicado pela natureza do serviço em si, que exige um processo de investigação antes de providências serem tomadas. O combate a roedores toma em média 49 dias, e a manutenção de árvores necessita de aproximadamente 42 dias. Por se tratar de intervenções biológicas e ambientais que frequentemente exigem inspeções especializadas, acompanhamento contínuo e agendamento de equipes operacionais pesadas, esses longos prazos refletem a complexidade inerente aos serviços prestados.

Chama a atenção o valor negativo do tempo médio de encerramento para os casos de ninhos de gansos (-0,08 dias). Ao investigar a causa dessa anomalia, percebe-se que 100% dos 372 chamados desta categoria foram finalizados com o motivo 'Closed automatically and sent to service group'. Isso indica a existência de uma regra de automação no sistema de chamados da prefeitura: no momento em que o cidadão registra uma solicitação desse tipo, o sistema encaminha o alerta imediatamente para um grupo especializado e encerra o protocolo no sistema principal no mesmo instante. Além de que existem solicitações onde a data de encerramento foi dada antes da de abertura da solicitação, o que pode indicar que um problema que ja foi solucionado foi solicitado a ser resolvido.

In [ ]:
(df_urban_forestry[df_urban_forestry['Service request type'].isin(['Canada Geese Nests Case'])]
    .groupby('Closure reason')['Department'].count()
    .reset_index()
    .set_axis(['Motivo do encerramento','Quantidade'],axis=1))

## Mapa de solicitações para manutenção de árvores

In [ ]:
import folium
from folium.plugins import HeatMap

df_validos_uf = df_urban_forestry[df_urban_forestry['Service request type'].isin(['City and Park Trees Maintenance Case'])]
df_validos_uf = df_validos_uf.dropna(subset=['Latitude','Longitude'])

mapa = folium.Map(
    location=[
        df_validos_uf['Latitude'].mean(),
        df_validos_uf['Longitude'].mean(),
    ],
    zoom_start=12,
)



dados_calor = df_validos_uf[['Latitude','Longitude']].values.tolist()


HeatMap(dados_calor, radius=12,blur=10,min_opacity=0.1).add_to(mapa)

mapa

In [ ]:
import matplotlib.image as mpimg

img = mpimg.imread(r"C:\Users\Erick\Desktop\Estudo\UFS\MINERAÇÃO DE DADOS\Mapas\mapa de calor de solicitação de controle de peste.png")


plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')  
plt.tight_layout()
plt.show()

# Departamento com maior número de solicitações (ENG - Sanitation Services)

O departamento com o maior número de solicitações e "ENG - Sanitation Services", a divisão responsável pela limpeza das ruas e coleta de lixo.

Todosos serviços prestados pelo departamento estão listados na tabela abaixo.

In [ ]:
df_sanitation_services = df_dept_top_3[df_dept_top_3['Department'].isin(['ENG - Sanitation Services'])]

tbl_num_ss = (df_sanitation_services['Service request type'].value_counts().
    reset_index()
    .set_axis(['Tipo de serviço solicitado','Quantidade'],axis=1)
    .sort_values(by='Quantidade',ascending=False))

(tbl_num_ss.style
   .set_caption("Serviçoes solicitados ao departamento ENG - Sanitation Services'")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

A análise dos tipos de chamados evidencia que a gestão e recolhimento de resíduos residenciais e orgânicos representam a principal demanda da população de Vancouver. Os problemas com a não coleta de lixeiras verdes de compostagem (Missed Green Bin Pickup Case, com 60.470 solicitações) e pedidos de novas lixeiras de lixo comum (Garbage Bin Request Case, com 59.578) lideram o volume absoluto de chamados.

## Tempo médio ate encerramento dos casos (ENG - Sanitation Services)

In [ ]:
tbl_media_ss = (df_sanitation_services.groupby('Service request type')['Dias ate encerramento'].mean()
    .reset_index()
    .set_axis(['Tipo de serviço solicitado','Dias até encerramento'],axis=1)
    .sort_values(by='Dias até encerramento',ascending=False)
    .head(10))

(tbl_media_ss.style
   .set_caption("Tempo médio até encerramento das solicitações na divisão Sanitation Services'")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

O tempo médio de encerramento dos casos dentro da divisão varia de acordo com a natureza do serviço solicitado. Consultas e investigações operacionais sobre o saneamento (Sanitation Operations Inquiry Case) exigem o maior tempo de resolução, levando em média 23 dias para serem concluídas. Solicitações referentes a alocaçã de lixeiras verdes leva entre 11 a 12 dias em média.

Na tabela acima podemos ver os 10 serviçoes com maior média de tempo até o encerramento.

## Os motivos mais frequentes para o encerramento dos casos no departamento (ENG -Sanitation Services)

In [ ]:
plt.figure(figsize=(7,6))
ax9 = (sns.countplot(y='Closure reason',data=df_sanitation_services,
               hue='Closure reason', legend=False,
               hue_order=df_sanitation_services['Closure reason'].value_counts().head(5).index,
               order=df_sanitation_services['Closure reason']
    .value_counts()
    .head(5).index))

for container in ax9.containers:
    ax9.bar_label(container,label_type='edge',color='black',fontweight='bold',
    labels=[f'{int(v.get_width()):} ({v.get_width()/df_sanitation_services["Closure reason"]
        .value_counts().head(5).sum():.1%})' for v in container])

plt.xlabel('Quantidade')

A grande maioria dos chamados deste departamento é finalizada com a execução efetiva do trabalho, totalizando 235.471 casos concluídos sob a categoria 'Service provided'. É relevante notar, contudo, o volume expressivo de chamados encerrados como desconhecido ('Unknown', com 25.590 ocorrências) ou por problema não encontrado/inacessível ('Issue not found or inaccessible', com 24.845 ocorrências). Esses números sinalizam eventuais falhas no detalhamento de endereços no momento do registro ou situações em que o resíduo descartado foi removido antes da chegada da equipe de coleta.

## Mapa de solicitações

No mapa abaixo podemos ver onde mais ocorrem casos de descarte de recicláveis pela cidade de Vancouver.

In [ ]:
df_validos_ss = df_sanitation_services[df_sanitation_services['Service request type'].isin(['Abandoned Recyclables Case'])]
df_validos_ss = df_validos_ss.dropna(subset=['Latitude','Longitude'])

mapa1 = folium.Map(
    location=[
        df_validos_ss['Latitude'].mean(),
        df_validos_ss['Longitude'].mean(),
    ],
    zoom_start=12,
)



dados_calor1 = df_validos_ss[['Latitude','Longitude']].values.tolist()

HeatMap(dados_calor1, radius=15,blur=10,min_opacity=0.1).add_to(mapa1)

mapa1

In [ ]:
# Carregar a imagem
img = mpimg.imread(r"C:\Users\Erick\Desktop\Estudo\UFS\MINERAÇÃO DE DADOS\Mapas\mapa de calor de reciclaveis descartados.png")

plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')  
plt.tight_layout()
plt.show()

É evidente que as solicitações de recolhimento de material reciclável descartado tem volume concentrado no centro de Vancouver (Downtown),além dos bairros vizinhos como Strathcona e Grandview-Woodland. A elevada densidade nesses pontos urbanos (destacados em cores quentes no mapa) correlaciona-se com áreas de maior adensamento populacional, zonas comerciais movimentadas e locais com expressivo fluxo diário de pedestres.

# Análise do departamento ENG - Streets Operations

Para a próxima parte do relatório resolvi trabalhar somente com o departamento de engenharia, especificamente na divisão de "Streets operations", responsáveis pela manutenção das vias públicas. Para tal, filtrei somente pelas observações do departamento.

In [ ]:
df_streets_ops = df[df['Department']=='ENG - Streets Operations']

Após o filtro, checamos quantas observações nós temos

In [ ]:
df_streets_ops.info()

Sabendo que temos 77302 observações, vamos ver quantas são missing.

In [ ]:
tbl_miss_street = df_streets_ops.isnull().sum().reset_index().set_axis(['Departamento - Divisão','Missing'],axis=1)
(tbl_miss_street.style
   .set_caption("Dados Missing do departamento ENG - Streets Operations")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

verificação da qualidade da base filtrada para a divisão de Streets Operations (77302 registros) mostra que as variáveis principais, como tipo de serviço, status, canal e timestamps de abertura, apresentam preenchimento completo. A maior concentração de dados ausentes ocorre nas variáveis de geolocalização e endereço, onde 11194 registros não possuem coordenadas de latitude e longitude. Essa ausência parcial não compromete a análise geral do setor.

## Análise exploratória

## Tipos de serviço solicitados

A principio é preciso ver quais tipos de solicitações a divisão de streets operations recebe e quais são os mais frequentes.

In [ ]:
#| include: false
tbl_tipo_sreet = (df_streets_ops['Service request type'].value_counts().reset_index()
    .set_axis(['Tipo de serviço solicitado','Quantidade de chamados'],axis=1))

(tbl_tipo_sreet .style
   .set_caption("Tipos de serviços solicitados ao departamento ENG - Streets Operations")
   .set_table_styles([{'selector':'caption','props':[('font-size','16px'),('font-weight','bold')]}]))

Na tabela acima temos todos os tipos de serviço solicitado a esse departamento, além da quantidade de cada um dos 12 tipos.

In [ ]:
plt.figure(figsize=(10,7))
ax = sns.countplot(y="Service request type",data=df_streets_ops,hue='Service request type',
             order=df_streets_ops["Service request type"].value_counts().index,palette='viridis')
plt.xlabel('Quantidade de chamados')

for container in ax.containers:
    ax.bar_label(container,label_type='edge',color='black',fontweight='bold',
                labels=[f'{int(v.get_width()):} ({v.get_width()/len(df_streets_ops):.1%})' for v in container])

plt.title('Ranking de solicitações mais frequentes',fontsize=16,fontweight='bold')  
plt.tight_layout()

O gráfico de ranking revela uma forte concentração das solicitações da população em problemas viários do dia a dia. As solicitações para reparo de buracos (Pothole Case) lideram de forma isolada, somando 19575 chamados. Em seguida, problemas de acúmulo e inundação de água na superfície das ruas (Street Surface Water Flooding Case, com 9412 chamados) e reparações gerais em vias (Street Repair Case, com 9400 chamados) aparecem com volumes expressivos. Juntas, essas três categorias representam a maioria absoluta de todas as demandas direcionadas à divisão de Streets Operations, evidenciando que a manutenção do asfalto e o escoamento de águas são as prioridades operacionais do município.

## Proporção de casos abertos e encerrados

In [ ]:
plt.figure(figsize=(7,4))
plt.ylabel('Quantidade')
ax13 = sns.countplot(x='Status',data=df_streets_ops,hue='Status')
for container in ax13.containers:
    ax13.bar_label(container,fontweight='bold',
                  labels=[f'{int(v.get_height()):} ({v.get_height()/len(df_streets_ops):.1%})' for v in container])
plt.title('Proporção de casos abertos e fechados',fontsize=16,fontweight='bold')    
plt.xticks(ticks=[0, 1],labels=['Fechado', 'Aberto'])
plt.tight_layout()

A análise da proporção de casos confirma a elevada eficiência operacional da equipe de Streets Operations. No gráfico geral de status, a barra indicativa de casos fechados domina amplamente o cenário, mostrando que o departamento consegue solucionar e encerrar a esmagadora maioria dos chamados recebidos.

In [ ]:
df_open_close = (pd.crosstab(df_streets_ops['Service request type'], df_streets_ops['Status'])
    .reset_index()
    .set_axis(['Tipo de serviço solicitado','Fechado','Aberto'],axis=1)
    .sort_values(by='Fechado',ascending=False))
(df_open_close.style
   .set_caption("Proporção de Casos Abertos e Fechados por Tipo de Serviço")
   .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]))

Ao observar a tabela por categoria, nota-se que mesmo nos serviços de alta demanda a quantidade de casos pendentes é baixa, como na categoria de buracos (19.448 fechados contra 127 abertos) e no reparo de calçadas (8.519 fechados contra 272 abertos). Isso demonstra a capacidade da divisão em processar as demandas sem gerar acúmulo de pendências.

## Motivos mais comuns para encerramento dos casos

In [ ]:
df_clsr_rsn_top10 = (df_streets_ops['Closure reason'].value_counts()
    .reset_index()
    .set_axis(['Motivo do fechamento','Quantidade'],axis=1).head(10))

In [ ]:
plt.figure(figsize=(10,7))
ax1 = sns.barplot(y='Motivo do fechamento',x='Quantidade',hue='Motivo do fechamento',data=df_clsr_rsn_top10)

total_top10 = df_clsr_rsn_top10['Quantidade'].sum()
for container in ax1.containers:
    ax1.bar_label(container,label_type='edge',color='black',fontweight='bold',
                 labels=[f'{int(v.get_width()):} ({v.get_width()/total_top10:.1%})' for v in container])

plt.title('Ranking dos motivos para encerramento dos casos',fontweight='bold',fontsize=16)
plt.tight_layout()

A distribuição das razões de fechamento demonstra que a maior parte dos chamados resulta na execução direta do trabalho. O motivo 'Service provided' (serviço prestado) é o predominante, cobrindo 44272 chamados do departamento. Além disso, 8959 casos foram encerrados sob a justificativa de ação futura planejada ('Further action has been planned') e 7755 foram encaminhados para outros grupos de serviço. Apenas 4530 casos foram classificados como analisados sem ação planejada ('Reviewed and no action planned'), o que reforça o compromisso do departamento em intervir na infraestrutura viária sempre que necessário.

## Tempo médio de encerramento dos casos

In [ ]:
#tirando o horario e o periodo da service request open timestamp
df_streets_ops['Service request open timestamp'] = df_streets_ops['Service request open timestamp'].apply(lambda x:x.split('T')[0])
#transofrmando em data
df_streets_ops['Service request open timestamp'] = pd.to_datetime(df_streets_ops['Service request open timestamp'])
df_streets_ops['Service request close date'] = pd.to_datetime(df_streets_ops['Service request close date'])

In [ ]:
#diferença dos dias
df_streets_ops['Dias ate encerramento'] = (
    df_streets_ops['Service request close date'] - df_streets_ops['Service request open timestamp'])
df_streets_ops[['Service request open timestamp','Service request close date','Dias ate encerramento']]

df_streets_ops['Dias ate encerramento'] = df_streets_ops['Dias ate encerramento'].dt.days 

In [ ]:
df_media_dias = (df_streets_ops.groupby('Service request type')['Dias ate encerramento'].mean()
    .reset_index()
    .set_axis(['Tipo de serviço solicitado','Tempo médio em dias'],axis=1)
    .sort_values(by='Tempo médio em dias',ascending=False))
(df_media_dias.style
   .set_caption("Tempo médio até encerramento dos serviços solicitados do departamento ENG - Streets Operations")
   .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}]))

O tempo de resposta do departamento varia significativamente conforme a complexidade técnica e a urgência de cada serviço. Solicitações de infraestrutura pesada registram os maiores prazos, a exemplo do reparo de ruas (Street Repair Case, com média de 18 dias), manutenção de pontes e estruturas (18 dias) e reparo de calçadas (12 dias). Por outro lado, atendimentos emergenciais e sazonais de segurança, como a remoção de neve e gelo em propriedades da cidade, são solucionados rapidamente em uma média de 2 dias.

In [ ]:
plt.figure(figsize=(10,7))
sns.boxplot(x='Dias ate encerramento',y='Service request type'
            ,hue='Service request type',palette='viridis',data=df_streets_ops,showfliers=False)

plt.xlabel('Dias até encerramento')
plt.title('Dias até encerramento para cada tipo de caso',fontsize=16,fontweight='bold')
plt.tight_layout()

## Tipos de vias onde mais frequentemente existem ocorrências

In [ ]:
df_streets_ops['Tipo de via'] = df_streets_ops['Address'].apply(lambda x:str(x).split(' ')) #splitando em lista de strings e criando a coluna

In [ ]:
def via_id(endereço):
    if 'AV' in endereço or 'AVE' in endereço:
        return 'Avenida'
    elif 'ST' in endereço:
        return 'Rua'
    return 'Outro' 

In [ ]:
df_streets_ops['Tipo de via'] = df_streets_ops['Tipo de via'].apply(via_id)
df_streets_ops['Tipo de via'].unique()

In [ ]:
plt.figure(figsize=(10,7))
ax3 = sns.countplot(x='Tipo de via',data=df_streets_ops,hue='Tipo de via',palette='viridis')
plt.ylabel('Quantidade')
for container in ax3.containers:
    ax3.bar_label(container,label_type='edge',color='black')

plt.title('Solicitações por tipo de via',fontsize=16,fontweight='bold')    

A comparação entre as categorias de vias mostra que as avenidas concentram o maior volume absoluto de solicitações (29440 casos), seguidas de perto pelas ruas locais (28022 casos).

## Frequência de casos de buraco, reparo da via, e reparo de calçada por tipo de via

In [ ]:
df_pot_repair = (df_streets_ops[df_streets_ops['Service request type'].isin(['Pothole Case','Street Repair Case','Sidewalk Repair Case'])]
    [['Service request type','Latitude','Longitude','Tipo de via']]) #obs somente de buraso e reparo

In [ ]:
plt.figure(figsize=(10,7))
ax5 = sns.countplot(x='Tipo de via',data=df_pot_repair,hue='Service request type',palette='viridis')
plt.ylabel('Quantidade')
for container in ax5.containers:
    ax5.bar_label(container,label_type='edge',color='black')

ax5.legend(title='Tipo de serviço solicitado')  
plt.title('Solicitações de reparo',fontsize=16,fontweight='bold')  

Ao cruzar esses dados com as solicitações de reparo, os buracos (Potholes) consolidam-se como o problema mais recorrente tanto em avenidas (8617 solicitações) quanto em ruas (7916 solicitações). A maior incidência de buracos nas avenidas é explicada pelo tráfego mais intenso e pelo fluxo contínuo de veículos pesados. Reparos em calçadas também são mais frequentes em avenidas.

## Locais da cidade com maior solicitação de serviço de reparo de ruas e buracos

In [ ]:
df_pot_repair = (df_streets_ops[df_streets_ops['Service request type'].isin(['Pothole Case','Street repair Case'])]
    [['Service request type','Latitude','Longitude']])

In [ ]:
import folium
from folium.plugins import MarkerCluster

mapa = folium.Map(location=[df_pot_repair['Latitude'].mean(), df_pot_repair['Longitude'].mean()],zoom_start=12)

mc = MarkerCluster().add_to(mapa)

df_validos = df_pot_repair.dropna(subset=['Latitude','Longitude'])

for _, row in df_validos.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Tipo: {row['Service request type']}" 
    ).add_to(mc)

mapa

In [ ]:
import matplotlib.image as mpimg

img = mpimg.imread(r"C:\Users\Erick\Desktop\Estudo\UFS\MINERAÇÃO DE DADOS\Mapas\mapa buracos.png")


plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')  
plt.tight_layout()
plt.show()

In [ ]:
df_validos = df_pot_repair.dropna(subset=['Latitude','Longitude'])


mapa = folium.Map(
    location=[df_validos['Latitude'].mean(), df_validos['Longitude'].mean()],
    zoom_start=12,
)

dados_calor = df_validos[['Latitude','Longitude']].values.tolist()

HeatMap(dados_calor,radius=15,blur=8,min_opacity=0.1).add_to(mapa)

mapa

In [ ]:
img = mpimg.imread(r"C:\Users\Erick\Desktop\Estudo\UFS\MINERAÇÃO DE DADOS\Mapas\Mapa de calor dos buracos.png")

plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off') 
plt.tight_layout()
plt.show()

A distribuição geoespacial apresentada no mapa de calor evidencia adensamentos críticos, indicados pelas áreas em vermelho e alaranjado, onde a malha viária exige maiores intervenções. Observa-se uma concentração significativa de chamados por buracos e reparos viários na região central e ao longo da faixa norte do município de Vancouver. Essa densidade de ocorrências reflete os principais corredores de circulação diária e rotas de transporte coletivo, que sofrem maior degradação e demandam rotas frequentes de manutenção preventiva e corretiva.

# Comparação entre as três divisões analisadas

In [ ]:
plt.figure(figsize=(10.9,6))
df_compr_geral = df[df['Department'].isin(['ENG - Streets Operations','ENG - Sanitation Services','PR - Urban Forestry'])]

anos_ordenados = sorted(df_compr_geral['Ano'].unique())

ax15 = sns.countplot(x='Ano',data=df_compr_geral,hue='Department',palette='viridis',order=anos_ordenados)




for container in ax15.containers:
    ax15.bar_label(container,fontweight='bold')

plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1))
plt.tight_layout()
plt.title('Volume de solicitações recebidas por ano pelos 3 departamentos analisados',fontsize=18)

A divisão de Saneamento (Sanitation Services) lidera disparado com a maioria das solicitações por ano, pois lida com serviços diários e contínuos de coleta de lixo. Streets Operations fica em um patamar intermediário, focado na manutenção do asfalto, e Urban Forestry tem o menor volume, mas demanda equipes muito especializadas o que resulta em um tempo médio para encerramento dos casos bem maior.

# Conclusões

A análise dos dados revela uma nítida assimetria na relação entre o volume de chamados, a complexidade técnica dos serviços e o tempo de resposta exigido de cada setor. A divisão de Sanitation Services sobressai-se com o maior volume, ultrapassando consistentemente a marca de setenta mil solicitações anuais entre 2022 e 2025. Este fluxo constante é acompanhado por uma elevada taxa de resolutividade, o que demonstra uma operação de rotina bem consolidada e eficiente no recolhimento de resíduos e na limpeza urbana. Contudo, a manutenção de volumes tão elevados ao longo dos anos sinaliza que a gestão atua essencialmente sobre os efeitos e não sobre as causas, indicando a necessidade de evoluir o foco operacional para políticas de conscientização comunitária e fiscalização do descarte irregular.

Em contrapartida, o departamento de Urban Forestry regista um volume significativamente menor de solicitações, situando-se entre dez mil e doze mil chamados anuais, mas apresenta o maior tempo médio para o encerramento dos casos. Esta lentidão não reflete necessariamente uma ineficiência administrativa, mas sim a elevada complexidade e o risco inerentes ao manejo de árvores em ambiente urbano, o que exige equipas especializadas, maquinário pesado, intervenções coordenadas com a rede elétrica e vistorias ambientais criteriosas.

Por sua vez, a divisão de Streets Operations lida com um volume intermediario de chamados por ano, predominantemente voltados para a manutenção de calçadas e a reparação de buracos. As intervenções neste setor sofrem um impacto direto do tráfego, nos quais a demora na resposta amplifica a degradação da via e multiplica o número de reclamações sobre o mesmo ponto. A concentração de chamados nos eixos viários principais evidencia um modelo de atuação predominantemente reativo.